# Process for **Accruals** Export for Cancer Center Support Grant (CCSG)
## Table of Contents
1. [Introduction](#introduction)
    - [Detailed Summary](#detailed-summary)
    - [Scope and Limitations](#scope-and-limitations)
2. [Setting Up the Environment](#setting-up-the-environment)
    1. [Dependencies](#21-install-dependencies)
    2. [Loading Environment Variables](#22-loading-environment-variables)
    3. [Oracle Database Connection Setup](#23-oracle-database-connection-setup)
3. [Implementing Business Rules](#3-implementing-business-rules)
    - [Protocol Export Type Classification](#protocol-export-type-classification)
    - [Subject Accrual Export Criteria](#subject-accrual-export-criteria)
    - [Summary Accrual Export Criteria](#summary-accrual-export-criteria)
4. [Workflow](#4-workflow)
    1. [Defining Parameters for Export](#defining-parameters-for-export)

### 1. Introduction
#### Detailed Summary
The National Cancer Institute (NCI) requires upload of Subject Accrual Information from Pariticipating Institutions to the Clinical Trials Reporting Program (CTRP) for NCI funded trials on a quarterly basis as part of the Cancer Center Support Grant (CCSG). This is typically done through CTRP's Manual Upload Service, where users manually upload batch files of subject accruals to the CTRP website.
CTRP offers an Macro Enabled Spreadsheet to simplify the process, however as the volume of trials and accruals increase this process can become more time consuming and prone to user error. This notebook offers a Proof of Concept and a programmatic framework for simplifying the export of accrual data from OnCore, a Clinical Trials Management System (CTMS), to CTRP.

#### How to Use

#### Scope and Limitations
- This notebook does not include the functionality to upload the generated batch files to CTRP. It only generates the batch files that are to be uploaded to CTRP. An API Key and Client ID/Secret are required to upload the generated batch files to CTRP. The process of obtaining these credentials is beyond the scope of this notebook.

### 2. Setting Up Environment
- #### 2.1. Install Dependencies
The notebook is dependent on Oracle Thick Client and Python dependencies. 

To satisfy this requirement, follow the steps below:

**2.1.1. Install Oracle Thick Client**

- Download Oracle Thick Client from [Oracle Website](https://download.oracle.com/otn_software/nt/instantclient/1932000/instantclient-basic-windows.x64-19.32.0.0.0dbru.zip).
- Extract the downloaded zip file to a desired location. 
- Add the path of the extracted directory to the `ORACLE_PATH` environment variable

**2.1.2. Install Python Dependencies**
- The notebook is dependent on the following packages:
```
python-dotenv
oracledb
pandas
pydantic
```

Run the following to install the packages
```
python -m pip install python-dotenv oracledb pandas pydatnic
```

#### 2.2. Loading Environment Variables
- The `.env` file is used to store sensitive information such as DB connection strings
    - Copy `.env.example` to `.env`
- Replace the environment variables from `.env` file.
    - Example:
    ``` 
    ORACLE_PATH="[ORACLE_PATH]"
    ORACLE_USERNAME="[ORACLE_DB_USER]"
    ORACLE_PASSWORD="[ORACLE_DB_PASSWORD]"
    ORACLE_HOST="[ORACLE_DB_HOST]"
    ORACLE_PORT="[ORACLE_DB_PORT]"
    ORACLE_SERVICE_NAME="[ORACLE_DB_SERVICE_NAME]"
    ```

In [ ]:
import os
import filecmp
if not os.path.exists('./.env'):
    import shutil
    shutil.copy('./.env.example', './.env')
    raise FileNotFoundError(""".env file not found. File has been created from .env.example. Please edit and fill in all environment variables then re-run this block""")
elif filecmp.cmp("./.env", "./.env.example", shallow=False):
    raise ValueError("Please update ./env file variables have not changed from the example")

# 2. Loading Environment Variables
from dotenv import load_dotenv
load_dotenv()

# Access Environment Variables
oracle_path = os.environ['ORACLE_PATH']
db_username = os.environ['ORACLE_USERNAME']
db_password = os.environ['ORACLE_PASSWORD']
db_host = os.environ['ORACLE_HOST']
db_port = os.environ['ORACLE_PORT']
db_service_name = os.environ['ORACLE_SERVICE_NAME']

#### 2.3. Oracle Database Connection Setup
- Setup Oracle Thick Client to connect to the Oracle OnCore Database
- Validate Connection

In [ ]:
# 3. Oracle DB Connection Setup
import oracledb

# Setup Oracle Thick Client
oracledb.init_oracle_client(lib_dir=oracle_path)

# Validating connection
from abc import ABC, abstractmethod
import pandas as pd
import pyarrow

class OnCoreExport(ABC):
    """
    Helper class that stores a query and uses the data to retrieve information from OnCore
    """
    def __init__(self, query: str):
        self.query = query

    def oracle_export_df(self, query_params: dict[str, str]=None) -> pd.DataFrame:
        try:
            if query_params == None:
                query_params = dict()
            query = self.query
            query_submit = query.format(**query_params)

            with oracledb.connect(
                user=db_username,
                password=db_password,
                dsn=f"{db_host}:{db_port}/{db_service_name}"
            ) as conn:
                odf = conn.fetch_df_all(statement=query_submit)
                return pyarrow.table(odf).to_pandas()
        except Exception as e:
            print(query_params, query)
            raise e
    @classmethod
    def query_from_file(cls, filename):
        with open(filename, "r") as f:
            return cls(f.read())
            
export_test = OnCoreExport("SELECT 1 FROM DUAL")
assert export_test.oracle_export_df().shape[1] > 0

## 3. Implementing Business Rules
Protocol characteristics are used to classify the **Export Type** for the accrual information (Subject / Summary / None). We use the information about the Principal Protocol Sponsor (Sponsor Type and whether the Institution is the Principal Sponsor) to determine if an Export is necessary.

Protocol -> Classification of Export Type -> Check if Export is Necessary

**Note** : Business Rules will vary by your Institution. Please work with the Subject Matter Expert and Stakeholders to identify the corresponding rules.

### 3.1. Business Rule Matrix
- #### Protocol Export Type Classification
    | Library | Investigator Initiated Trial | Data Table 4 Type | Accrual Export Type |
    |-------------|-------|-------------------|-------------|
    | Oncology | Yes | Observational | Summary |
    | Oncology | No | Observational | Summary |
    | Oncology | Yes | Interventional | Subject |   
    | Oncology | No | Interventional | Summary |

In [ ]:
# Protocol Export Classification
pcl_rules_df = pd.DataFrame([
    {'LIBRARY': 'Oncology', 'IIT': 'Yes', 'DT4_TYPE': 'Observational', 'EXPORT_TYPE': 'Summary'}
    , {'LIBRARY': 'Oncology', 'IIT': 'No', 'DT4_TYPE': 'Observational', 'EXPORT_TYPE': 'Summary'}
    , {'LIBRARY': 'Oncology', 'IIT': 'Yes', 'DT4_TYPE': 'Interventional', 'EXPORT_TYPE': 'Subject'}
    , {'LIBRARY': 'Oncology', 'IIT': 'No', 'DT4_TYPE': 'Observational', 'EXPORT_TYPE': 'Summary'}
])

pcl = OnCoreExport.query_from_file("./ProtocolInformation.sql")
pcl_df = pcl.oracle_export_df()

pcl_export_df = pd.merge(left=pcl_df, right=pcl_rules_df, how="left", on=["LIBRARY", "IIT", "DT4_TYPE"])
pcl_export_df["P_SPON_IDS"] = pcl_export_df["P_SPON_IDS"].str.split(",").apply(lambda x: [int(v) for v in x] if type(x) == list else [])
pcl_export_df["EVAL_SPON_TYPES"] = pcl_export_df["EVAL_SPON_TYPES"].str.split(",")

# Principal Sponsor(s)
p_spon = OnCoreExport.query_from_file("./PrincipalSponsor.sql")
principal_sponsor_df = p_spon.oracle_export_df()
sponsor_ids = principal_sponsor_df["SPONSOR"].values
pcl_export_df["INSTITUTION_PRINCIPAL_SPONSOR"] = pcl_export_df["P_SPON_IDS"].apply(lambda x: any(v in sponsor_ids for v in x))

The following are the business rules to determine if the institution is responsible CTRP upload if it falls within the export criteria. There resulting dataframe is `pcl_criteria` which provides the overall scope

- #### Subject Accrual Export Criteria
    | Principal Sponsor Type | Principal Sponsor is Institution | Principal Sponsor is NOT Institution |
    |------------------------------------|--------------|------|
    | National | None | None |
    | External Peer Review | Export Subject Accrual | None |
    | Institutional | Export Subject Accrual | None |
    | Industry | None | None |
<br>

- #### Summary Accrual Export Criteria
    | Principal Sponsor Type | Principal Sponsor is Institution | Principal Sponsor is NOT Institution |
    |------------------------------------|--------------|------|
    | National | None | None |
    | External Peer Review | Export Summary Accrual | None |
    | Institutional | Export Summary Accrual | None |
    | Industry | Export Summary Accrual | Export Summary Accrual |
    


In [ ]:
# Declaration of Export Requirements based on Sponsors for both Summary and Subject Accrual
subject_accrual_rules = pd.DataFrame([{
        "Sponsor Type": "External Peer Review",
        "Institution Principal Sponsor": True,
        "Export": True
    },
    {
        "Sponsor Type": "Institutional",
        "Institution Principal Sponsor": True,
        "Export": True
    }])

summary_accrual_rules = pd.DataFrame([
    {
        "Sponsor Type": "External Peer Review",
        "Institution Principal Sponsor": True,
        "Export": True
    },
    {
        "Sponsor Type": "Institutional",
        "Institution Principal Sponsor": True,
        "Export": True
    },
    {
        "Sponsor Type": "Industry",
        "Institution Principal Sponsor": True,
        "Export": True
    },
    {
        "Sponsor Type": "Industry",
        "Institution Principal Sponsor": False,
        "Export": True
    }
])

pcl_export_df_join = ["EVAL_SPON_TYPES", "INSTITUTION_PRINCIPAL_SPONSOR"]
accrual_df_join = ["Sponsor Type", "Institution Principal Sponsor"]
explode_def = {"left": ["EVAL_SPON_TYPES", "P_SPON_IDS"]}

# Joins with the business rules and applies True if it meets any of the conditions
def assign_export_decision(left: pd.DataFrame, right: pd.DataFrame,
    left_on: list, right_on: list,
    explode_dict:dict=None):
    temp_left = left.copy()
    if explode_dict.get("left", []):
        temp_left = left.explode(explode_dict["left"])
    
    temp_right = right
    if explode_dict.get("right", []):
        temp_right = right.explode(explode_dict["right"])

    return pd.merge(left=temp_left, left_on=left_on,
        right=temp_right, right_on=right_on,
        how="left").drop(right_on, axis=1).groupby("PROTOCOL_ID")["Export"].transform(lambda x: x.any())

final_pcl_export_df = pcl_export_df.copy()
final_pcl_export_df["Export"] = assign_export_decision(pcl_export_df, summary_accrual_rules
    , left_on=pcl_export_df_join, right_on=accrual_df_join, explode_dict=explode_def)

final_pcl_export_df["Export"] = final_pcl_export_df["Export"] | assign_export_decision(pcl_export_df, subject_accrual_rules
    , left_on=pcl_export_df_join, right_on=accrual_df_join, explode_dict=explode_def)

export_valid = (~final_pcl_export_df.EXPORT_TYPE.isna()) & (final_pcl_export_df.Export)

### Helper Classes
Standardizing the data model and parameters for export
#### ExportType
Enum denoting export type Subject or Summary
#### TimePeriod
Contains a start date and end date for a time range with a helper function to check if data is between two dates.
#### FilterCriteria
Contains lists for NCI Number, National Clinical Trials (NCT) Number, Protocol Number to check for exclusion or inclusion
#### FilterParameters
Contains the filter criteria for Inclusion or Exclusion
#### ExportParameters
Defining the time period scope to look for accruals, time period to export accruals across, protocol_ids of Studies that are in scope from the business rules, any additional Inclusion/Exclusion Criteria, and a export_name for the directory and zip file.

In [ ]:
from enum import Enum
import zipfile
from datetime import date
from typing import Any
from pydantic import BaseModel, Field, model_validator, field_validator
from pydantic_core import PydanticUndefined
from collections import defaultdict
from os import makedirs

class ExportType(Enum):
    SUBJECT=1
    SUMMARY=2

class TimePeriod(BaseModel):
    start_date: date=Field(default=date(1900, 1, 1))
    end_date: date=Field(default_factory=date.today)

    def between_range(self, date_col: pd.Series) -> pd.Series:
        return (date_col.dt.date >= self.start_date) & (date_col.dt.date <= self.end_date)

    def describe(self):
        return f"{self.start_date.strftime("%Y-%m-%d")} to {self.end_date.strftime("%Y-%m-%d")}"
    
    @model_validator(mode="before")
    @classmethod
    def define_default(cls, data: Any) -> Any:
        if isinstance(data, dict):
            for field_name, field_item in cls.model_fields.items():
                if field_name not in data or data[field_name] is None:
                    default_value = field_item.get_default(call_default_factory=True)
                    data[field_name] = default_value
        return data

    @model_validator(mode="after")
    def start_before_end(self):
        if self.start_date > self.end_date:
            raise ValueError("Start is after the end date")
        return self

class FilterCriteria(BaseModel):
    NCI_ID: list[object] = Field(default_factory=list)
    NCT_ID: list[object] = Field(default_factory=list)
    PROTOCOL_NO: list[object] = Field(default_factory=list)

class FilterParameters(BaseModel):
    include: FilterCriteria = Field(default_factory=FilterCriteria)
    exclude: FilterCriteria = Field(default_factory=FilterCriteria)

class ExportParameters(BaseModel):
    accrual_scope: TimePeriod
    export_scope: TimePeriod
    protocol_ids: list[int] = Field(default_factory=list)
    filter_parameters: FilterParameters = Field(default_factory=FilterParameters)
    export_name: str = ""

# Generic Export class for CTRP
class CTRPExport(ABC):
    def __init__(self, oncore_export: OnCoreExport, export_type:ExportType):
        self.oncore_export = oncore_export
        self.export_type = export_type

    def retrieve_data(self, protocol_id_list:list)->pd.DataFrame:
        protocol_criteria = ""
        if protocol_id_list:
            protocol_criteria = f"AND pcl.protocol_id IN ({', '.join(str(protocol_id) for protocol_id in protocol_id_list)})"
        return self.oncore_export.oracle_export_df({"protocol_criteria": protocol_criteria})
    
    def handle_params(self, dataframe, filter_parameters):
        def eval_params(dataframe, params) -> pd.Series:
            result = pd.Series(False, index=dataframe.index)
            for col_name, data in params:
                result |= dataframe[col_name].isin(data)
            return result

        conditional = pd.Series(True, index=dataframe.index)

        exclude_condition = eval_params(dataframe, filter_parameters.exclude)
        include_condition = eval_params(dataframe, filter_parameters.include)

        conditional = (conditional | include_condition) & ~exclude_condition
        
        return dataframe[conditional]

    @abstractmethod
    def transform(self, dataframe, export_parameters: ExportParameters):
        pass    

    def handle_export(self, dataframe: pd.DataFrame, export_parameters: ExportParameters):
        file_prefix = date.today().strftime("%Y-%m-%d") + self.export_type.name if not export_parameters.export_name else export_parameters.export_name
        out_list = []
        fail_list = []
        os.makedirs(file_prefix, exist_ok=True)

        for nci_id in dataframe.NCI_ID.dropna().unique():
            filename = file_prefix + "." + str(nci_id) + ".txt"
            rel_path = file_prefix + "\\" + filename
            try:
                data_scope = dataframe.loc[dataframe.NCI_ID == nci_id, :]
                data_scope = self.get_export_data(data_scope, export_parameters.export_scope)
                self.write_nci_file(data_scope, rel_path, str(nci_id))
                out_list.append(rel_path)
            except Exception as e:
                print(e)
                fail_list.append(rel_path)
        return out_list, fail_list
    
    def handle_zip(self, file_prefix, file_list):
        zip_name = f"{file_prefix}.zip"
        with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
            for f_name in file_list:
                file_path = f_name
                if os.path.exists(file_path):
                    zipf.write(file_path, arcname=f_name)
        return zip_name

    @abstractmethod
    def get_export_data(self, dataframe: pd.DataFrame, export_scope: TimePeriod):
        pass

    @abstractmethod
    def write_nci_file(self, dataframe: pd.DataFrame, filename:str, nci_id: str):
        pass

    def export(self, export_parameters: ExportParameters):
        df = self.retrieve_data(export_parameters.protocol_ids)
        res_df = self.transform(df, export_parameters)
        out_list, fail_list = self.handle_export(res_df, export_parameters)

        file_prefix = date.today().strftime("%Y-%m-%d") + self.export_type.name if not export_parameters.export_name else export_parameters.export_name
        zip_name = self.handle_zip(file_prefix, out_list)
        print(f"Zip File Created: {zip_name}")

#### CTRPExport
Base class that takes in an OnCoreExport (Query with ability to take in format String) and ExportType (to denote the Export Type). Establishes the base pipeline for the export:
Data Retrieval -> Cleaning and Filtering -> Exporting Data

#### SubjectExport
Granularity: Individual Study - Subject - Race.
There are non-mandatory fields in the CTMS that are often empty (Sequence Number, Zip Code, Birth Date, ICD Code, Country Code) which are required for export to CTRP.

#### SummaryExport
Granularity: Study - Enrollment Site - Month
Rolls up accruals for the Study producing a running sum by the last day of the month. Enrollment Sites are identified by PO Ids.


In [ ]:
class SubjectExport(CTRPExport):
    def __init__(self, oncore_export: OnCoreExport):
        super().__init__(oncore_export, ExportType.SUBJECT)

    def transform(self, dataframe, export_parameters: ExportParameters) -> pd.Dataframe:
        filtered_df = self.handle_params(dataframe, export_parameters.filter_parameters)
        protocol_id_scope = filtered_df.loc[export_parameters.accrual_scope.between_range(filtered_df["ON_STUDYDATE"]), "PROTOCOL_ID"].unique()
        res_df = filtered_df.loc[filtered_df["PROTOCOL_ID"].isin(protocol_id_scope), :].copy()
        print(f"Number of protocols {export_parameters.accrual_scope.describe()}: {len(protocol_id_scope)}")

        # Fill in NA Logic (All below are Mandatory)
        # - Sequence_Number - %Y-Q
        res_df.loc[res_df.SEQUENCE_NUMBER.isna(), "NUM"] = res_df.sort_values('ON_STUDYDATE').loc[res_df.SEQUENCE_NUMBER.isna(), :]\
            .groupby(["PROTOCOL_ID", res_df["ON_STUDYDATE"].dt.to_period("Q")])\
                ["ON_STUDYDATE"].rank(method="dense").astype("Int64")
        res_df.loc[res_df.SEQUENCE_NUMBER.isna(), "SEQUENCE_NUMBER"] = res_df.ON_STUDYDATE.dt.strftime("%Y-Q") + res_df.ON_STUDYDATE.dt.quarter.astype(str) + "-" + res_df.NUM.astype(str).str.zfill(3)

        # - ZIP (NULL) -> 99999
        res_df.loc[res_df["ZIP"].isna(), "ZIP"] = "99999"
        
        # - BIRTH_DATE (NULL) -> 1990-06-15
        res_df["BIRTH_DATE"] = res_df["BIRTH_DATE"].fillna("1990-06-15")

        # - ICD_CODE (NULL) -> Z1000
        res_df["ICD_CODE"] = res_df["ICD_CODE"].fillna("Z1000")

        # - COUNTRY_CODE (NULL + 5 digit Zip) -> US
        res_df.loc[res_df["ZIP"].str.len() == 5, "COUNTRY_CODE"] = res_df.loc[res_df["ZIP"].str.len() == 5, "COUNTRY_CODE"].fillna("US")

        return res_df.sort_values(["NCI_ID", "ON_STUDYDATE", "GENDER", "SEQUENCE_NUMBER"], ascending=[True, True, False, False])

    def write_nci_file(self, dataframe: pd.DataFrame, filename: str, nci_id: str)->str:
        try:
            lines = []
            with open(filename, "w+") as f:
                first_line = f'"COLLECTIONS","{nci_id}",,,,,,,,,"1"\n'
                lines.append(first_line)
                seq_set = set()
                race_lines = []

                for row_index, row in dataframe.iterrows():
                    seq_no = row.SEQUENCE_NUMBER
                    zip_code = row.ZIP
                    country_code = row.COUNTRY_CODE
                    birthday = row.BIRTH_DATE.strftime("%Y%m")
                    gender = row.GENDER
                    ethnicity = row.ETHNICITY
                    onstudy_date = row.ON_STUDYDATE.strftime("%Y%m%d")
                    po_id = row.PO_ID
                    icd_code = row.ICD_CODE

                    race = row.RACE

                    if seq_no not in seq_set:
                        line = f'"PATIENTS","{nci_id}","{seq_no}","{zip_code}","{country_code}","{birthday}","{gender}","{ethnicity}",,"{onstudy_date}",,"{po_id}",,,,,,,,,,"{icd_code}",,\n'
                        lines.append(line)

                    race_line = f'"PATIENT_RACES","{nci_id}","{seq_no}","{race}"\n'
                    race_lines.append(race_line)

                    seq_set.add(seq_no)
                lines = lines + race_lines
                f.writelines(lines)
            return filename
        except Exception as e:
            raise e

    def get_export_data(self, dataframe: pd.DataFrame, export_scope: TimePeriod):
        return dataframe.loc[export_scope.between_range(dataframe["ON_STUDYDATE"]), :]

class SummaryExport(CTRPExport):
    def __init__(self, oncore_export: OnCoreExport):
        super().__init__(oncore_export, ExportType.SUMMARY)    

    def transform(self, dataframe, export_parameters: ExportParameters):
        filtered_df = self.handle_params(dataframe, export_parameters.filter_parameters)
        protocol_id_scope = filtered_df.loc[export_parameters.accrual_scope.between_range(filtered_df["CUTOFF_DATE"]), "PROTOCOL_ID"].unique()
        res_df = filtered_df.loc[filtered_df["PROTOCOL_ID"].isin(protocol_id_scope), :].copy()
        print(f"Number of protocols {export_parameters.accrual_scope.describe()}: {len(protocol_id_scope)}")
        
        res_df.TOTAL_ACCRUALS = res_df.TOTAL_ACCRUALS.astype(int)

        # Finding scope
        res_df = res_df.sort_values(["NCI_ID", "PO_ID", "CUTOFF_DATE"])
        cols_out = ["NCI_ID", "PO_ID", "TOTAL_ACCRUALS", "CUTOFF_DATE"]

        res_df["LINE_OUT"] = '"ACCRUAL_COUNT",' + res_df.loc[:, cols_out].astype(str).agg(lambda x: ','.join(f'"{v}"' for v in x), axis=1) + "\n"
        return res_df
    
    def write_nci_file(self, dataframe: pd.DataFrame, filename: str, nci_id: str)->str:
        try:
            lines = []
            with open(filename, "w+") as f:
                first_line = f'"COLLECTIONS","{nci_id}",,,,,,,,,"1"\n'
                lines = [first_line]
                lines = lines + dataframe["LINE_OUT"].to_list()

                f.writelines(lines)
            
            return filename
        except Exception as e:
            raise e
        
    def get_export_data(self, dataframe: pd.DataFrame, export_scope: TimePeriod):
        return dataframe.loc[export_scope.between_range(dataframe["CUTOFF_DATE"]), :]


## 4. Workflow
### Defining Parameters for Export
From the Protocols that have been deemed in scope by the business rules, define a start and end date to identify Protocols that have accrued a subject during that range.

Scope Date Range:
- Criteria for identifying Protocols based on accruals within the Scope Date Range

Export Date Range:
- The range to export Accruals for Protocols

Include / Exclude
- Explicitly designate additional Studies that should be included or excluded based on NCI / NCT / Protocol Number

In [ ]:
import datetime as dt
# Parameters
#######
scope_start_date = None 
scope_end_date = None

export_start_date = None
export_end_date = None

include = {} 
exclude = {}
#######

def date_helper(date_input: object, base_value):
    if date_input is None:
        return base_value
    elif isinstance(date_input, date):
        return date_input
    elif isinstance(date_input, str):
        return pd.to_datetime(date_input)

# Defaults to Todays date and the first day of the Quarter
scope_end_date = date_helper(scope_end_date, (pd.to_datetime(dt.date.today()).to_period("Q") - 1).end_time.normalize())
scope_start_date = date_helper(scope_start_date, scope_end_date.to_period('Q').start_time)

accrual_scope = TimePeriod(start_date=scope_start_date, end_date=scope_end_date)
print(f"Scope is any studies with Accruals between {accrual_scope.describe()}")

# Defaults to 1900-01-01 to Today if undefined
export_scope = TimePeriod(start_date=export_start_date, end_date=export_end_date) 
print(f"Exports accruals for studies between {export_scope.describe()}")


filter_params = FilterParameters(include=include, exclude=exclude)

The Export Criteria identifies the overall scope of Protocols to include in this export. This will be narrowed down by the accrual scope, inclusion, and exclusion criteria. Only accruals within the export range will be considered for export. Afterwards, a zip file will be generated in the CTRP defined format for subject and summary respectively.

In [ ]:
subject_script = OnCoreExport.query_from_file("SubjectAccrual-CCSG.sql")
subject_export = SubjectExport(subject_script)


subject_export_criteria = export_valid & (final_pcl_export_df["EXPORT_TYPE"] == "Subject")
subject_export_ids = final_pcl_export_df.loc[subject_export_criteria, "PROTOCOL_ID"].to_list()

subject_export_params = ExportParameters(
    accrual_scope=accrual_scope,
    export_scope=export_scope,
    filter_parameters=filter_params,
    protocol_ids=subject_export_ids,
    export_name="2026 - Quarter 2 - Subject")

subject_export.export(subject_export_params)

In [ ]:
summary_script = OnCoreExport.query_from_file("SummaryAccrual-CCSG.sql")
summary_export = SummaryExport(summary_script)

summary_export_criteria = export_valid & (final_pcl_export_df["EXPORT_TYPE"] == "Summary")
summary_export_ids = final_pcl_export_df.loc[summary_export_criteria, "PROTOCOL_ID"].to_list()

summary_export_params = ExportParameters(
    accrual_scope=accrual_scope,
    export_scope=export_scope,
    filter_parameters=filter_params,
    protocol_ids=summary_export_ids,
    export_name="2026 - Quarter 2 - Summary")

summary_export.export(summary_export_params)